# Day 82: Data Versioning and DVC

Welcome to Day 82 of the 100 Days of Machine Learning! Today we dive into one of the most critical yet often overlooked aspects of MLOps: **Data Version Control (DVC)**.

## Introduction

In traditional software development, Git has become the standard for version control. However, machine learning projects face unique challenges:

- **Large datasets** that can't be stored efficiently in Git
- **Data transformations** that need to be tracked and reproduced
- **Model artifacts** that are too large for standard version control
- **Experiment tracking** across different data versions and model configurations

This is where **DVC (Data Version Control)** comes in. DVC is an open-source tool that brings Git-like versioning capabilities to machine learning projects, handling data files, models, and ML pipelines.

## Why Data Versioning Matters

Imagine you trained a model three months ago that performed exceptionally well. Today, you want to reproduce those results, but you can't remember:
- Which version of the training data you used
- What preprocessing steps were applied
- Which hyperparameters were chosen
- What the exact model architecture was

Without proper data versioning, reproducibility becomes nearly impossible. In production ML systems, this can lead to:
- Failed model deployments
- Inability to debug model performance issues
- Compliance and audit problems
- Wasted time re-running experiments

## Learning Objectives

By the end of this lesson, you will:
1. Understand the importance of data versioning in ML workflows
2. Learn the core concepts and architecture of DVC
3. Implement data versioning for ML projects
4. Create reproducible ML pipelines using DVC
5. Track experiments and compare different data versions

Let's get started!

## Core Concepts: DVC Architecture

### What is DVC?

DVC (Data Version Control) is built on top of Git and follows similar principles:

- **Git tracks code** → DVC tracks data and models
- **Git uses `.git/` directory** → DVC uses `.dvc/` directory  
- **Git has commits** → DVC has data/model versions tied to Git commits
- **Git has remotes (GitHub, GitLab)** → DVC has remotes (S3, GCS, Azure, local)

### How DVC Works

Instead of storing large files in Git, DVC:
1. Stores actual data files in a **remote storage** (S3, Google Cloud, etc.)
2. Tracks **metadata files** (`.dvc` files) in Git
3. Uses **content-addressable storage** (like Git objects)
4. Maintains **dependency graphs** for ML pipelines

### Mathematical Foundation: Content Addressing

DVC uses MD5 hashing to create content addresses:

$$\text{hash} = \text{MD5}(\text{file\_content})$$

This means:
- Same content → Same hash → No duplicate storage
- Different content → Different hash → New version
- Efficient storage through deduplication

### DVC Pipeline Structure

A typical ML pipeline with DVC looks like:

```
Raw Data (DVC) → Preprocessing (Pipeline Stage) → 
Processed Data (DVC) → Training (Pipeline Stage) → 
Model (DVC) → Evaluation (Pipeline Stage) → Metrics (DVC)
```

Each stage can have:
- **Dependencies** (`deps`): Input files/data
- **Outputs** (`outs`): Generated files/models
- **Parameters** (`params`): Hyperparameters from config files
- **Metrics** (`metrics`): Performance measurements

### Key DVC Commands

```bash
# Initialize DVC in a Git repository
dvc init

# Track a large file or directory
dvc add data/large_dataset.csv

# Configure remote storage
dvc remote add -d myremote s3://mybucket/dvcstore

# Push data to remote storage
dvc push

# Pull data from remote storage
dvc pull

# Reproduce a pipeline
dvc repro

# Compare experiments
dvc metrics diff
```

### The `.dvc` File Format

When you run `dvc add data.csv`, DVC creates a `data.csv.dvc` file:

```yaml
outs:
- md5: a304afb96060aad90176268345e10355
  size: 37891850
  path: data.csv
```

This small metadata file goes into Git, while the actual data goes to DVC remote storage.

### Benefits Over Traditional Approaches

| Challenge | Traditional Approach | DVC Approach |
|-----------|---------------------|-------------|
| Large files in Git | Repository bloat, slow clones | Lightweight metadata in Git |
| Data versioning | Manual naming (data_v1, data_v2) | Automatic version tracking |
| Reproducibility | Document steps manually | Pipeline as code |
| Storage costs | Duplicate full datasets | Deduplicated storage |
| Team collaboration | Share files manually | Push/pull like Git |


## Python Setup and Simulation

Since DVC is primarily a command-line tool, we'll use Python to:
1. Simulate data versioning concepts
2. Demonstrate file hashing and content addressing
3. Create a simple ML pipeline tracking system
4. Visualize version history and dependencies

Let's start by importing the necessary libraries:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
import json
from datetime import datetime
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
np.random.seed(42)

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## Demonstration 1: Content-Addressable Storage

Let's understand how DVC uses hashing to track file versions. We'll create a simple content-addressable storage system:

In [ ]:
def compute_hash(data):
    """Compute MD5 hash of data (simulating DVC's approach)"""
    if isinstance(data, str):
        data = data.encode('utf-8')
    elif isinstance(data, pd.DataFrame):
        data = data.to_csv(index=False).encode('utf-8')
    elif isinstance(data, np.ndarray):
        data = data.tobytes()
    return hashlib.md5(data).hexdigest()

def create_dvc_metadata(filepath, data, version):
    """Create a .dvc metadata file (simulated)"""
    file_hash = compute_hash(data)
    size = len(str(data))
    
    metadata = {
        'outs': [{
            'md5': file_hash,
            'size': size,
            'path': filepath
        }],
        'version': version,
        'timestamp': datetime.now().isoformat()
    }
    return metadata

# Example: Track different versions of a dataset
print("Simulating DVC data tracking...\n")

# Version 1: Original dataset
data_v1 = pd.DataFrame({
    'feature1': np.random.randn(100),
    'feature2': np.random.randn(100),
    'label': np.random.randint(0, 2, 100)
})
metadata_v1 = create_dvc_metadata('training_data.csv', data_v1, 'v1')
print("Version 1 metadata:")
print(json.dumps(metadata_v1, indent=2))

# Version 2: Updated dataset (more samples)
data_v2 = pd.DataFrame({
    'feature1': np.random.randn(150),
    'feature2': np.random.randn(150),
    'label': np.random.randint(0, 2, 150)
})
metadata_v2 = create_dvc_metadata('training_data.csv', data_v2, 'v2')
print("\nVersion 2 metadata:")
print(json.dumps(metadata_v2, indent=2))

# Check if versions are different
print("\n" + "="*50)
print(f"Hash changed: {metadata_v1['outs'][0]['md5'] != metadata_v2['outs'][0]['md5']}")
print(f"Size changed: {metadata_v1['outs'][0]['size']} → {metadata_v2['outs'][0]['size']}")
print("\nConclusion: DVC detects data changes through hash comparison!")

## Demonstration 2: ML Pipeline Tracking

Let's create a simple ML pipeline and track its dependencies, similar to how DVC tracks pipeline stages:

In [ ]:
class SimplePipelineTracker:
    """Simple pipeline tracker simulating DVC pipeline functionality"""
    
    def __init__(self):
        self.stages = {}
        self.history = []
    
    def add_stage(self, name, deps, outs, params=None):
        """Add a pipeline stage"""
        stage = {
            'name': name,
            'deps': deps,
            'outs': outs,
            'params': params or {},
            'executed': False,
            'timestamp': None
        }
        self.stages[name] = stage
        return stage
    
    def execute_stage(self, name, metrics=None):
        """Mark a stage as executed"""
        if name in self.stages:
            self.stages[name]['executed'] = True
            self.stages[name]['timestamp'] = datetime.now().isoformat()
            self.stages[name]['metrics'] = metrics or {}
            self.history.append({
                'stage': name,
                'timestamp': self.stages[name]['timestamp'],
                'metrics': metrics
            })
    
    def get_pipeline_status(self):
        """Get status of all pipeline stages"""
        status = pd.DataFrame([
            {
                'Stage': name,
                'Executed': stage['executed'],
                'Dependencies': len(stage['deps']),
                'Outputs': len(stage['outs']),
                'Timestamp': stage['timestamp'] or 'N/A'
            }
            for name, stage in self.stages.items()
        ])
        return status

# Create a pipeline
tracker = SimplePipelineTracker()

# Define stages
tracker.add_stage(
    name='data_preparation',
    deps=['raw_data.csv'],
    outs=['processed_data.csv'],
    params={'test_size': 0.2, 'random_state': 42}
)

tracker.add_stage(
    name='feature_engineering',
    deps=['processed_data.csv'],
    outs=['features.csv', 'feature_importance.json']
)

tracker.add_stage(
    name='model_training',
    deps=['features.csv'],
    outs=['model.pkl'],
    params={'n_estimators': 100, 'max_depth': 10}
)

tracker.add_stage(
    name='model_evaluation',
    deps=['model.pkl', 'features.csv'],
    outs=['metrics.json', 'confusion_matrix.png']
)

print("Pipeline stages defined!")
print("\nPipeline Status:")
print(tracker.get_pipeline_status().to_string(index=False))

## Demonstration 3: Running an ML Experiment with Version Tracking

Now let's run a complete ML experiment and track everything - data, parameters, and results:

In [ ]:
# Generate synthetic dataset
print("Stage 1: Data Preparation")
print("="*50)

n_samples = 1000
n_features = 5

X = np.random.randn(n_samples, n_features)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

data_df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(n_features)])
data_df['label'] = y

data_hash = compute_hash(data_df)
print(f"Generated {n_samples} samples with {n_features} features")
print(f"Data hash: {data_hash}")
print(f"Label distribution: {pd.Series(y).value_counts().to_dict()}")

# Mark stage as complete
tracker.execute_stage('data_preparation', metrics={'n_samples': n_samples})

# Feature engineering (simulated)
print("\nStage 2: Feature Engineering")
print("="*50)

# Add interaction features
data_df['feature_interaction'] = data_df['feature_0'] * data_df['feature_1']
feature_hash = compute_hash(data_df)
print(f"Added interaction features")
print(f"Feature data hash: {feature_hash}")
print(f"Total features: {data_df.shape[1] - 1}")

tracker.execute_stage('feature_engineering', metrics={'n_features': data_df.shape[1] - 1})

# Model training
print("\nStage 3: Model Training")
print("="*50)

X_train, X_test, y_train, y_test = train_test_split(
    data_df.drop('label', axis=1), 
    data_df['label'], 
    test_size=0.2, 
    random_state=42
)

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# Compute model "hash" (simplified)
model_params = str(model.get_params())
model_hash = compute_hash(model_params)
print(f"Model trained: RandomForestClassifier")
print(f"Model configuration hash: {model_hash[:16]}...")

tracker.execute_stage('model_training', metrics={'model_type': 'RandomForest'})

# Model evaluation
print("\nStage 4: Model Evaluation")
print("="*50)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

tracker.execute_stage('model_evaluation', metrics={'accuracy': accuracy})

print("\n" + "="*50)
print("Pipeline execution complete!")
print("\nFinal Pipeline Status:")
print(tracker.get_pipeline_status().to_string(index=False))

## Visualization: DVC Workflow and Pipeline Dependencies

Let's visualize how DVC tracks versions and pipeline dependencies:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: Version history timeline
ax1 = axes[0]
versions = ['v1.0', 'v1.1', 'v1.2', 'v2.0', 'v2.1']
dates = pd.date_range(start='2024-01-01', periods=len(versions), freq='W')
accuracies = [0.82, 0.85, 0.87, 0.89, 0.91]
data_sizes = [1000, 1500, 2000, 3000, 3500]

color = ['blue' if v.startswith('v1') else 'green' for v in versions]

ax1.scatter(range(len(versions)), accuracies, s=[size/5 for size in data_sizes], 
           c=color, alpha=0.6, edgecolors='black', linewidth=2)
ax1.plot(range(len(versions)), accuracies, 'k--', alpha=0.3)

for i, (v, acc, size) in enumerate(zip(versions, accuracies, data_sizes)):
    ax1.annotate(f'{v}\n{size} samples', xy=(i, acc), xytext=(0, 10),
                textcoords='offset points', ha='center', fontsize=9)

ax1.set_xlabel('Version', fontsize=12, fontweight='bold')
ax1.set_ylabel('Model Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('DVC Version History: Model Performance Over Time', fontsize=14, fontweight='bold')
ax1.set_xticks(range(len(versions)))
ax1.set_xticklabels(versions)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0.78, 0.94)

# Right plot: Pipeline dependency graph
ax2 = axes[1]

# Define pipeline stages and positions
stages = {
    'Raw Data': (0, 3),
    'Preprocessing': (1, 3),
    'Features': (2, 3),
    'Train': (3, 4),
    'Evaluate': (3, 2),
    'Model': (4, 4),
    'Metrics': (4, 2)
}

dependencies = [
    ('Raw Data', 'Preprocessing'),
    ('Preprocessing', 'Features'),
    ('Features', 'Train'),
    ('Features', 'Evaluate'),
    ('Train', 'Model'),
    ('Evaluate', 'Metrics')
]

# Draw nodes
for stage, (x, y) in stages.items():
    is_data = stage in ['Raw Data', 'Features', 'Model', 'Metrics']
    color = 'lightblue' if is_data else 'lightcoral'
    ax2.scatter(x, y, s=2000, c=color, alpha=0.7, edgecolors='black', linewidth=2, zorder=3)
    ax2.text(x, y, stage, ha='center', va='center', fontsize=10, fontweight='bold', zorder=4)

# Draw edges
for src, dst in dependencies:
    x1, y1 = stages[src]
    x2, y2 = stages[dst]
    ax2.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', lw=2, color='gray', alpha=0.6),
                zorder=1)

ax2.set_xlim(-0.5, 4.5)
ax2.set_ylim(1.5, 4.5)
ax2.set_xlabel('Pipeline Flow', fontsize=12, fontweight='bold')
ax2.set_title('DVC Pipeline: Stage Dependencies', fontsize=14, fontweight='bold')
ax2.set_xticks([])
ax2.set_yticks([])
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['bottom'].set_visible(False)
ax2.spines['left'].set_visible(False)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='lightblue', edgecolor='black', label='Data/Artifacts'),
    Patch(facecolor='lightcoral', edgecolor='black', label='Processing Stages')
]
ax2.legend(handles=legend_elements, loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

print("\nVisualization Explanation:")
print("- Left: Shows how model performance improves across DVC versions")
print("  (bubble size = dataset size, color = major version)")
print("- Right: Shows pipeline stage dependencies tracked by DVC")
print("  (blue = data/models, red = processing stages)")

## Demonstration 4: Comparing Experiments

One of DVC's most powerful features is comparing different experiments. Let's simulate this:

In [ ]:
# Simulate multiple experiments with different configurations
experiments = []

configs = [
    {'n_estimators': 50, 'max_depth': 5, 'data_version': 'v1'},
    {'n_estimators': 100, 'max_depth': 10, 'data_version': 'v1'},
    {'n_estimators': 150, 'max_depth': 15, 'data_version': 'v1'},
    {'n_estimators': 100, 'max_depth': 10, 'data_version': 'v2'},
    {'n_estimators': 100, 'max_depth': 10, 'data_version': 'v3'},
]

print("Running multiple experiments...\n")

for i, config in enumerate(configs, 1):
    # Simulate different data versions
    n_samples = 1000 if config['data_version'] == 'v1' else (
        1500 if config['data_version'] == 'v2' else 2000
    )
    
    X = np.random.randn(n_samples, 5)
    y = (X[:, 0] + X[:, 1] > 0).astype(int)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    model = RandomForestClassifier(
        n_estimators=config['n_estimators'],
        max_depth=config['max_depth'],
        random_state=42
    )
    model.fit(X_train, y_train)
    
    accuracy = accuracy_score(y_test, model.predict(X_test))
    
    experiment = {
        'experiment_id': f'exp_{i}',
        'data_version': config['data_version'],
        'n_estimators': config['n_estimators'],
        'max_depth': config['max_depth'],
        'n_samples': n_samples,
        'accuracy': accuracy
    }
    experiments.append(experiment)
    
    print(f"Experiment {i}: Data={config['data_version']}, "
          f"n_estimators={config['n_estimators']}, "
          f"max_depth={config['max_depth']}, "
          f"Accuracy={accuracy:.4f}")

# Create comparison dataframe
exp_df = pd.DataFrame(experiments)

print("\n" + "="*70)
print("Experiment Comparison (like 'dvc metrics diff'):")
print("="*70)
print(exp_df.to_string(index=False))

# Find best experiment
best_exp = exp_df.loc[exp_df['accuracy'].idxmax()]
print("\n" + "="*70)
print(f"Best Experiment: {best_exp['experiment_id']}")
print(f"  Data Version: {best_exp['data_version']}")
print(f"  Configuration: n_estimators={best_exp['n_estimators']}, max_depth={best_exp['max_depth']}")
print(f"  Accuracy: {best_exp['accuracy']:.4f}")

Let's visualize the experiment comparison:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: Accuracy comparison
ax1 = axes[0]
colors = ['red' if v == 'v1' else 'blue' if v == 'v2' else 'green' 
          for v in exp_df['data_version']]
bars = ax1.bar(exp_df['experiment_id'], exp_df['accuracy'], color=colors, alpha=0.7, edgecolor='black')

# Highlight best experiment
best_idx = exp_df['accuracy'].idxmax()
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)

ax1.set_xlabel('Experiment ID', fontsize=12, fontweight='bold')
ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('Experiment Comparison: Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_ylim(min(exp_df['accuracy']) - 0.02, max(exp_df['accuracy']) + 0.02)
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for i, (exp, acc) in enumerate(zip(exp_df['experiment_id'], exp_df['accuracy'])):
    ax1.text(i, acc + 0.002, f'{acc:.3f}', ha='center', fontsize=9)

# Right: Impact of hyperparameters
ax2 = axes[1]
scatter = ax2.scatter(
    exp_df['n_estimators'], 
    exp_df['max_depth'], 
    s=exp_df['accuracy'] * 1000,
    c=exp_df['n_samples'],
    cmap='viridis',
    alpha=0.6,
    edgecolors='black',
    linewidth=2
)

ax2.set_xlabel('n_estimators', fontsize=12, fontweight='bold')
ax2.set_ylabel('max_depth', fontsize=12, fontweight='bold')
ax2.set_title('Hyperparameter Space Exploration\n(size=accuracy, color=data size)', 
             fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Dataset Size', fontsize=10)

# Add experiment labels
for i, row in exp_df.iterrows():
    ax2.annotate(row['experiment_id'], 
                xy=(row['n_estimators'], row['max_depth']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=8)

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("- Left plot: Direct accuracy comparison across all experiments")
print("- Right plot: Shows relationship between hyperparameters and performance")
print("- DVC makes it easy to track and compare all these variations!")

## Hands-On Activity: Build Your Own DVC-Style Version Tracker

Now it's your turn! Let's build a simplified version tracking system that mimics DVC's core functionality:

### Task
Create a `DataVersionTracker` class that can:
1. Track different versions of a dataset
2. Compute and store metadata (hash, size, timestamp)
3. Compare versions and detect changes
4. Roll back to previous versions

### Starter Code

In [ ]:
class DataVersionTracker:
    """A simplified DVC-like data version tracker"""
    
    def __init__(self, project_name):
        self.project_name = project_name
        self.versions = {}  # Store version metadata
        self.data_store = {}  # Simulate remote storage
        self.current_version = None
    
    def add_version(self, version_name, data):
        """Add a new data version"""
        # Compute hash
        data_hash = compute_hash(data)
        
        # Create metadata
        metadata = {
            'hash': data_hash,
            'size': len(str(data)),
            'timestamp': datetime.now().isoformat(),
            'shape': data.shape if hasattr(data, 'shape') else None
        }
        
        # Store version
        self.versions[version_name] = metadata
        self.data_store[data_hash] = data  # Store actual data by hash
        self.current_version = version_name
        
        print(f"✓ Added version '{version_name}'")
        print(f"  Hash: {data_hash}")
        print(f"  Size: {metadata['size']} bytes")
        if metadata['shape']:
            print(f"  Shape: {metadata['shape']}")
    
    def get_version(self, version_name):
        """Retrieve a specific version"""
        if version_name not in self.versions:
            print(f"✗ Version '{version_name}' not found")
            return None
        
        metadata = self.versions[version_name]
        data = self.data_store[metadata['hash']]
        print(f"✓ Retrieved version '{version_name}'")
        return data
    
    def compare_versions(self, version1, version2):
        """Compare two versions"""
        if version1 not in self.versions or version2 not in self.versions:
            print("✗ One or both versions not found")
            return
        
        meta1 = self.versions[version1]
        meta2 = self.versions[version2]
        
        print(f"\nComparing '{version1}' vs '{version2}':")
        print("=" * 50)
        print(f"Hash changed: {meta1['hash'] != meta2['hash']}")
        print(f"Size: {meta1['size']} → {meta2['size']} ({meta2['size'] - meta1['size']:+d} bytes)")
        
        if meta1['shape'] and meta2['shape']:
            print(f"Shape: {meta1['shape']} → {meta2['shape']}")
    
    def list_versions(self):
        """List all versions"""
        print(f"\nProject: {self.project_name}")
        print(f"Current version: {self.current_version}")
        print("\nAll versions:")
        print("=" * 70)
        
        for name, meta in self.versions.items():
            marker = "*" if name == self.current_version else " "
            print(f"{marker} {name}: {meta['timestamp']} (hash: {meta['hash'][:8]}...)")

# Example usage
print("Creating DataVersionTracker...\n")
tracker = DataVersionTracker("customer_churn_prediction")

# Version 1: Initial dataset
data_v1 = pd.DataFrame({
    'customer_id': range(100),
    'age': np.random.randint(18, 70, 100),
    'tenure': np.random.randint(1, 120, 100),
    'churn': np.random.randint(0, 2, 100)
})
tracker.add_version('v1.0_initial', data_v1)

print("\n" + "-"*70 + "\n")

# Version 2: Added more customers
data_v2 = pd.DataFrame({
    'customer_id': range(200),
    'age': np.random.randint(18, 70, 200),
    'tenure': np.random.randint(1, 120, 200),
    'churn': np.random.randint(0, 2, 200)
})
tracker.add_version('v2.0_expanded', data_v2)

print("\n" + "-"*70 + "\n")

# Version 3: Added feature engineering
data_v3 = data_v2.copy()
data_v3['age_tenure_ratio'] = data_v3['age'] / (data_v3['tenure'] + 1)
tracker.add_version('v3.0_feature_eng', data_v3)

print("\n" + "="*70)

# List all versions
tracker.list_versions()

print("\n" + "="*70)

# Compare versions
tracker.compare_versions('v1.0_initial', 'v3.0_feature_eng')

print("\n" + "="*70)

# Retrieve old version
print("\nRetrieving old version...")
old_data = tracker.get_version('v1.0_initial')
print(f"Retrieved data shape: {old_data.shape}")
print(f"\nFirst 5 rows:\n{old_data.head()}")

## Key Takeaways

Congratulations! You've learned about Data Version Control (DVC) and its importance in MLOps. Here are the main points:

### 1. **Why DVC Matters**
   - Large ML datasets and models can't be efficiently stored in Git
   - Reproducibility is critical in production ML systems
   - Teams need to collaborate on data just like they collaborate on code
   - Experiment tracking prevents wasted effort and enables comparison

### 2. **Core DVC Concepts**
   - **Content-addressable storage**: Files identified by hash, enabling deduplication
   - **Lightweight metadata**: Only small `.dvc` files go into Git
   - **Remote storage**: Actual data stored in S3, GCS, Azure, etc.
   - **Pipeline as code**: ML workflows defined in `dvc.yaml`

### 3. **DVC Workflow**
   ```bash
   dvc init              # Initialize DVC in Git repo
   dvc add data.csv      # Track large file
   git add data.csv.dvc  # Commit metadata
   dvc push              # Upload to remote storage
   dvc pull              # Download from remote
   dvc repro             # Reproduce pipeline
   ```

### 4. **Benefits Over Alternatives**
   - Better than manual versioning (data_v1, data_v2, etc.)
   - More efficient than storing large files in Git LFS
   - Integrates seamlessly with existing Git workflows
   - Open-source and storage-agnostic

### 5. **Best Practices**
   - Version control everything: data, code, models, configs
   - Use meaningful version names/tags
   - Define clear pipeline stages with dependencies
   - Track metrics for experiment comparison
   - Document data lineage and transformations

### 6. **When to Use DVC**
   - ✅ Datasets larger than 100MB
   - ✅ Multiple data versions or experiments
   - ✅ Team collaboration on ML projects
   - ✅ Need for reproducible pipelines
   - ✅ Production ML systems

### 7. **Real-World Impact**
   - **Reproducibility**: Recreate any experiment from history
   - **Collaboration**: Team members work on same data versions
   - **Debugging**: Track down what changed between model versions
   - **Compliance**: Audit trail for regulated industries
   - **Efficiency**: Deduplicated storage saves costs

### Next Steps
To continue your MLOps journey:
1. Install DVC and try it on a real project
2. Set up remote storage (start with local, then try cloud)
3. Define a multi-stage ML pipeline
4. Integrate with CI/CD (GitHub Actions, GitLab CI)
5. Explore DVC's experiment tracking features
6. Learn about model registry and deployment tools (MLflow, Kubeflow)

## Further Resources

### Official Documentation
1. **[DVC Official Documentation](https://dvc.org/doc)** - Comprehensive guide to all DVC features
2. **[DVC Tutorial](https://dvc.org/doc/start)** - Get started with hands-on examples
3. **[DVC Use Cases](https://dvc.org/doc/use-cases)** - Real-world applications and best practices

### Articles and Tutorials
4. **[How to Version Control Your Machine Learning Data](https://realpython.com/python-data-version-control/)** - Real Python tutorial
5. **[MLOps: Continuous Delivery and Automation in ML](https://cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning)** - Google Cloud guide
6. **[Data Versioning in Machine Learning Projects](https://neptune.ai/blog/data-versioning-in-machine-learning)** - Neptune.ai blog

### Videos and Courses
7. **[DVC YouTube Channel](https://www.youtube.com/c/DVC_org)** - Official tutorials and talks
8. **[Made With ML - MLOps Course](https://madewithml.com/)** - Comprehensive MLOps curriculum

### Tools and Alternatives
9. **[Git LFS](https://git-lfs.github.com/)** - Git Large File Storage (alternative approach)
10. **[MLflow](https://mlflow.org/)** - Complementary tool for experiment tracking
11. **[Pachyderm](https://www.pachyderm.com/)** - Data versioning with containerized pipelines
12. **[LakeFS](https://lakefs.io/)** - Git-like version control for data lakes

### Research Papers
13. **[Versioning for End-to-End Machine Learning Pipelines](https://dl.acm.org/doi/10.1145/3329486.3329492)** - Academic perspective on data versioning
14. **[Hidden Technical Debt in Machine Learning Systems](https://papers.nips.cc/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html)** - NIPS paper on ML engineering challenges

### Community
15. **[DVC Discord](https://dvc.org/chat)** - Active community for questions and discussions
16. **[r/MLOps](https://www.reddit.com/r/mlops/)** - Reddit community for MLOps topics

---

**Congratulations on completing Day 82!** You now have the foundational knowledge to implement robust data versioning in your ML projects. Tomorrow, we'll explore model monitoring and drift detection in production systems.